In [1]:
import os
import os
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, UnidentifiedImageError
from diffusers import UNet2DModel, DDPMScheduler
import torch.optim as optim
from tqdm import tqdm
import matplotlib.pyplot as plt

/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from diffusers import DDPMScheduler
from torch import nn
from torch import optim

# Define the ConditionalUNet model as you shared earlier
class ConditionalUNet(nn.Module):
    def __init__(self, num_classes=14):
        super().__init__()
        self.unet = UNet2DModel(
            sample_size=256, in_channels=3, out_channels=3,
            layers_per_block=2, block_out_channels=(128, 256, 512, 512),
            down_block_types=("DownBlock2D", "DownBlock2D", "DownBlock2D", "AttnDownBlock2D"),
            up_block_types=("AttnUpBlock2D", "UpBlock2D", "UpBlock2D", "UpBlock2D")
        )
        self.label_embedding = torch.nn.Linear(num_classes, 512)
        self.label_proj = torch.nn.Linear(512, 3)

    def forward(self, x, t, labels):
        label_embedding = self.label_embedding(labels)
        label_embedding = self.label_proj(label_embedding)
        label_embedding = label_embedding[:, :, None, None]
        x = x + label_embedding
        return self.unet(x, t)

# Load the trained model
checkpoint_path = "/mnt/Internal/MedImage/CheXpert Dataset//Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt/model_epoch_89.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the model architecture
model = ConditionalUNet().to(device)

# Load the state dictionary into the model
state_dict = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state_dict)

# Set the model to evaluation mode
model.eval()

# Initialize the noise scheduler (same as in training)
noise_scheduler = DDPMScheduler(num_train_timesteps=2000)

# Disease indices mapping
disease_indices = {
    "No Finding": 0,
    "Enlarged Cardiomediastinum": 1,
    "Cardiomegaly": 2,
    "Lung Opacity": 3,
    "Lung Lesion": 4,
    "Edema": 5,
    "Consolidation": 6,
    "Pneumonia": 7,
    "Atelectasis": 8,
    "Pneumothorax": 9,
    "Pleural Effusion": 10,
    "Pleural Other": 11,
    "Fracture": 12,
    "Support Devices": 13
}

# Define the function to generate images for a specific disease
@torch.no_grad()
def generate_images_for_disease(disease_name, num_images=1):
    disease_vector = [0] * 14
    disease_index = disease_indices[disease_name]
    disease_vector[disease_index] = 1  # Set the disease index to 1
    
    # Create a tensor for the disease label
    labels = torch.tensor(disease_vector, dtype=torch.float32).unsqueeze(0).to(device)

    # Generate a batch of images with the condition
    sample_noise = torch.randn(num_images, 3, 256, 256).to(device)
    for t in reversed(range(noise_scheduler.num_train_timesteps)):
        sample_noise = noise_scheduler.step(
            model(sample_noise, torch.tensor([t] * num_images).to(device), labels).sample,
            t,
            sample_noise
        ).prev_sample
    return sample_noise

# Function to generate and save images based on the filtered DataFrame and specific disease label
def generate_and_save_images(df_filtered, disease_name, output_base_dir, start_index=17001, end_index=18000):
    # Generate images for the specified disease
    with torch.autocast("cuda"):
        for i, row in df_filtered.iloc[start_index:end_index].iterrows():
            # Extract original image path from the DataFrame (assuming 'Path' column contains the path)
            original_path = row['Path']  # Replace 'Path' with the correct column name if needed

            # Extract patient ID, study ID, and view ID from the path
            path_parts = original_path.split('/')
            patient_id = path_parts[-3]  # patientXXX
            study_id = path_parts[-2]  # studyX
            view_id = path_parts[-1]  # viewX_frontal.jpg

            # Generate synthetic X-ray for the specified disease
            images = generate_images_for_disease(disease_name, num_images=1)
            
            # Normalize and save the generated image
            generated_image = images[0].cpu().detach().numpy()
            generated_image = (generated_image - generated_image.min()) / (generated_image.max() - generated_image.min())  # Normalize to 0-1 range

            # Create corresponding directory structure for generated images, including the patient-specific subdirectory
            generated_image_path = os.path.join(output_base_dir, "train", patient_id, study_id, view_id)
            os.makedirs(os.path.dirname(generated_image_path), exist_ok=True)

            # Save image
            plt.imsave(generated_image_path, generated_image.transpose(1, 2, 0), cmap="gray")
            print(f"Generated and saved: {generated_image_path}")

    print("The process is completed")

# Example usage:
# Load the CSV file into a DataFrame
df_filtered = pd.read_csv("/mnt/Internal/MedImage/merged_dataset-Copy1.csv")

# Filter rows where the 'Frontal/Lateral' column contains 'frontal'
df_filtered = df_filtered[df_filtered['Frontal/Lateral'].str.contains('frontal', case=False, na=False)]

# Replace -1 with 1 and NaN values with 0
df_filtered.replace(-1, 1, inplace=True)
df_filtered.replace(np.nan, 0, inplace=True)

# Ensure the output directory exists
output_base_dir = "/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt/"
os.makedirs(output_base_dir, exist_ok=True)

# Specify the disease you want to generate images for
disease_name = "Enlarged Cardiomediastinum"
# Generate and save the images for the specified disease and filtered DataFrame
generate_and_save_images(df_filtered, disease_name, output_base_dir, start_index=23001, end_index=24000)

/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/diffusers/configuration_utils.py:140: FutureWarning: Accessing config attribute `num_train_timesteps` directly via 'DDPMScheduler' object attribute is deprecated. Please access 'num_train_timesteps' over 'DDPMScheduler's config object instead, e.g. 'scheduler.config.num_train_timesteps'.
  deprecate("direct config name access", "1.0.0", deprecation_message, standard_warn=False)


Generated and saved: /mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt/train/patient07067/study7/view1_frontal.jpg
Generated and saved: /mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt/train/patient07067/study5/view1_frontal.jpg
Generated and saved: /mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt/train/patient07067/study10/view1_frontal.jpg
Generated and saved: /mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt/train/patient07067/study16/view1_frontal.jpg
Generated and saved: /mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt/train/patient07067/study4/view1_frontal.jpg
Generated and saved: /mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation

In [5]:
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from diffusers import DDPMScheduler
from torch import nn
from torch import optim

# Define the ConditionalUNet model as you shared earlier
class ConditionalUNet(nn.Module):
    def __init__(self, num_classes=14):
        super().__init__()
        self.unet = UNet2DModel(
            sample_size=256, in_channels=3, out_channels=3,
            layers_per_block=2, block_out_channels=(128, 256, 512, 512),
            down_block_types=("DownBlock2D", "DownBlock2D", "DownBlock2D", "AttnDownBlock2D"),
            up_block_types=("AttnUpBlock2D", "UpBlock2D", "UpBlock2D", "UpBlock2D")
        )
        self.label_embedding = torch.nn.Linear(num_classes, 512)
        self.label_proj = torch.nn.Linear(512, 3)

    def forward(self, x, t, labels):
        label_embedding = self.label_embedding(labels)
        label_embedding = self.label_proj(label_embedding)
        label_embedding = label_embedding[:, :, None, None]
        x = x + label_embedding
        return self.unet(x, t)

# Load the trained model
checkpoint_path = "/mnt/Internal/MedImage/CheXpert Dataset//Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt/model_epoch_89.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the model architecture
model = ConditionalUNet().to(device)

# Load the state dictionary into the model
state_dict = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state_dict)

# Set the model to evaluation mode
model.eval()

# Initialize the noise scheduler (same as in training)
noise_scheduler = DDPMScheduler(num_train_timesteps=2000)

# Disease indices mapping
disease_indices = {
    "No Finding": 0,
    "Enlarged Cardiomediastinum": 1,
    "Cardiomegaly": 2,
    "Lung Opacity": 3,
    "Lung Lesion": 4,
    "Edema": 5,
    "Consolidation": 6,
    "Pneumonia": 7,
    "Atelectasis": 8,
    "Pneumothorax": 9,
    "Pleural Effusion": 10,
    "Pleural Other": 11,
    "Fracture": 12,
    "Support Devices": 13
}

# Define the function to generate images for a specific disease
@torch.no_grad()
def generate_images_for_disease(disease_name, num_images=1):
    disease_vector = [0] * 14
    disease_index = disease_indices[disease_name]
    disease_vector[disease_index] = 1  # Set the disease index to 1
    
    # Create a tensor for the disease label
    labels = torch.tensor(disease_vector, dtype=torch.float32).unsqueeze(0).to(device)

    # Generate a batch of images with the condition
    sample_noise = torch.randn(num_images, 3, 256, 256).to(device)
    for t in reversed(range(noise_scheduler.num_train_timesteps)):
        sample_noise = noise_scheduler.step(
            model(sample_noise, torch.tensor([t] * num_images).to(device), labels).sample,
            t,
            sample_noise
        ).prev_sample
    return sample_noise

# Function to generate and save images based on the filtered DataFrame and specific disease label
def generate_and_save_images(df_filtered, disease_name, output_base_dir, start_index=24001, end_index=25000):
    # Generate images for the specified disease
    with torch.autocast("cuda"):
        for i, row in df_filtered.iloc[start_index:end_index].iterrows():
            # Extract original image path from the DataFrame (assuming 'Path' column contains the path)
            original_path = row['Path']  # Replace 'Path' with the correct column name if needed

            # Extract patient ID, study ID, and view ID from the path
            path_parts = original_path.split('/')
            patient_id = path_parts[-3]  # patientXXX
            study_id = path_parts[-2]  # studyX
            view_id = path_parts[-1]  # viewX_frontal.jpg

            # Generate synthetic X-ray for the specified disease
            images = generate_images_for_disease(disease_name, num_images=1)
            
            # Normalize and save the generated image
            generated_image = images[0].cpu().detach().numpy()
            generated_image = (generated_image - generated_image.min()) / (generated_image.max() - generated_image.min())  # Normalize to 0-1 range

            # Create corresponding directory structure for generated images, including the patient-specific subdirectory
            generated_image_path = os.path.join(output_base_dir, "train", patient_id, study_id, view_id)
            os.makedirs(os.path.dirname(generated_image_path), exist_ok=True)

            # Save image
            plt.imsave(generated_image_path, generated_image.transpose(1, 2, 0), cmap="gray")
            print(f"Generated and saved: {generated_image_path}")

    print("The process is completed")

# Example usage:
# Load the CSV file into a DataFrame
df_filtered = pd.read_csv("/mnt/Internal/MedImage/merged_dataset-Copy1.csv")

# Filter rows where the 'Frontal/Lateral' column contains 'frontal'
df_filtered = df_filtered[df_filtered['Frontal/Lateral'].str.contains('frontal', case=False, na=False)]

# Replace -1 with 1 and NaN values with 0
df_filtered.replace(-1, 1, inplace=True)
df_filtered.replace(np.nan, 0, inplace=True)

# Ensure the output directory exists
output_base_dir = "/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt/"
os.makedirs(output_base_dir, exist_ok=True)

# Specify the disease you want to generate images for
disease_name = "Cardiomegaly"
# Generate and save the images for the specified disease and filtered DataFrame
generate_and_save_images(df_filtered, disease_name, output_base_dir, start_index=24001, end_index=25000)

/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/diffusers/configuration_utils.py:140: FutureWarning: Accessing config attribute `num_train_timesteps` directly via 'DDPMScheduler' object attribute is deprecated. Please access 'num_train_timesteps' over 'DDPMScheduler's config object instead, e.g. 'scheduler.config.num_train_timesteps'.
  deprecate("direct config name access", "1.0.0", deprecation_message, standard_warn=False)


Generated and saved: /mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt/train/patient07397/study11/view1_frontal.jpg
Generated and saved: /mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt/train/patient07397/study7/view1_frontal.jpg
Generated and saved: /mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt/train/patient07397/study12/view1_frontal.jpg
Generated and saved: /mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt/train/patient07397/study5/view1_frontal.jpg
Generated and saved: /mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt/train/patient07397/study1/view1_frontal.jpg
Generated and saved: /mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation

In [9]:
# how to send files from one serveer to another
#  rsync -avz -e "ssh -p 56730" "merged_dataset-Copy1.csv" dawood@202.39.11.1:/mnt/Internal/MedImage/Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt/"
#rsync -avz -e "ssh -p 56730" "/mnt/backup1/Dawood's Data/CheXpert Dataset/Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt/train" dawood@202.39.11.1:"/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt"


In [ ]:
#22000 to 23000 = SD - Completed
#23001 to 24000 = "Enlarged Cardiomediastinum" - Completed
#24001 to 25000 = "Cardiomegaly" - Completed
#25001 to 26000 = "Lung Opacity" 
#26001 to 27000 =  "Lung Lesion" 
#27001 to 28000 = "Edema" 
#28001 to 29000 = "Consolidation" 
#29001 to 30000 = Pneumonia 
#30001 to 31000 = "Atelectasis" 
#31001 to 32000 =  "Pneumothorax" 
#32001 to 33000 =  "Pleural Effusion"
#33001 to 34000 = "Pleural Other"
#34001 to 35000 = "Fracture"
#35001 to 36000 = "Support Devices"